# Chapter 16 Lab — Nearest Neighbours

Can a new house inherit its label from nearby houses? What happens when the number of features grows? Read [`blog.md`](<blog.md>) first.

## Prediction

Predict the nearest class for `[3, 11]`, what feature scaling will do, and whether nearest/farthest distance contrast will grow or shrink with dimension.

In [ ]:
import numpy as np

X_train = np.array([[2,8],[2,10],[4,16],[5,18],[3,20]], dtype=float)
y_train = np.array(['Apartment','Apartment','Villa','Villa','Farmhouse'])
query = np.array([3,11], dtype=float)
distances = np.sqrt(((X_train - query) ** 2).sum(axis=1))
print(distances)
assert y_train[np.argmin(distances)] == 'Apartment'

## Mathematics

Euclidean distance: $d(x,q)=\sqrt{\sum_j(x_j-q_j)^2}$. k-NN predicts the majority label among the $k$ smallest distances.

In [ ]:
# Manual distance from [3,11] to A=[2,8]
manual_distance = np.sqrt((3-2)**2 + (11-8)**2)
assert np.isclose(manual_distance, np.sqrt(10))

# 3-NN vote
nearest3 = np.argsort(distances)[:3]
values, counts = np.unique(y_train[nearest3], return_counts=True)
prediction = values[np.argmax(counts)]
assert prediction == 'Apartment'

In [ ]:
def knn_predict(X_train, y_train, query, k=3):
    d = np.sqrt(((X_train - query) ** 2).sum(axis=1))
    idx = np.argsort(d)[:k]
    values, counts = np.unique(y_train[idx], return_counts=True)
    return values[np.argmax(counts)]

assert knn_predict(X_train, y_train, query, k=3) == 'Apartment'

In [ ]:
import matplotlib.pyplot as plt

for label in np.unique(y_train):
    pts = X_train[y_train == label]
    plt.scatter(pts[:,0], pts[:,1], label=label)
plt.scatter(query[0], query[1], marker='*', s=180, label='query')
plt.xlabel('rooms')
plt.ylabel('area (hundreds sq ft)')
plt.legend()
plt.show()

In [ ]:
# Scaling experiment: square feet are much larger numerically than room count
X_raw = np.array([[2,800],[3,1000],[4,1600],[5,1800]], dtype=float)
q_raw = np.array([3,1100], dtype=float)
raw_dist = np.sqrt(((X_raw - q_raw) ** 2).sum(axis=1))

mu = X_raw.mean(axis=0)
std = X_raw.std(axis=0)
X_scaled = (X_raw - mu) / std
q_scaled = (q_raw - mu) / std
scaled_dist = np.sqrt(((X_scaled - q_scaled) ** 2).sum(axis=1))
print('raw:', raw_dist)
print('scaled:', scaled_dist)

In [ ]:
# Change only the dimension and observe nearest/farthest contrast
rng = np.random.default_rng(7)
contrasts = []
for d in [2, 10, 50, 100]:
    pts = rng.random((500, d))
    q = rng.random(d)
    dist = np.sqrt(((pts - q) ** 2).sum(axis=1))
    contrasts.append((d, (dist.max() - dist.min()) / dist.min()))
print(contrasts)

## Observe

Scaling can change which point is nearest. As dimension increases, nearest and farthest distances tend to become relatively less separated in random data.

## Explain

Distance adds contributions from every feature. More dimensions mean more terms and rapidly growing neighbourhood volume requirements.

In [ ]:
# Level 4–5 challenge
# YOUR CODE HERE
# Compare k = 1, 3, 15 on a synthetic classification dataset.
# Then remove one irrelevant feature and see how the decision changes.

## Reflection

- [ ] I can compute and interpret Euclidean distance.
- [ ] I understand why scaling changes neighbours.
- [ ] I can explain what k controls.
- [ ] I can explain why high-dimensional local search becomes difficult.

Next: replace geometric voting with a sequence of learned questions.